# OLMo-2-1B × opc-sft-stage2 leaderboard — long-horizon (r=64, 9000 steps, ~225M tokens)

Long-horizon characterization from `docs/notes/polar_product/tight_chord_paper_plan.md` (Phase L). Cell: OLMo-2-1B × `opc-sft-stage2` (4 sub-configs concat, 436k docs → 150k packed slots @ seq=2048) × r=64 × global_batch=16 (batch=4 × accum=4) × packed_v1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell. `max_steps=9000` ≈ 225M unique content tokens (~295M slot-tokens at 75% fill).

Phase L LR sweep, seed=0, three optimizer arms. AdamW + chord-tight each pool three sub-sweeps (original L1, packed-v1.1 repack of same η values, η-extension); chord-tight-clean κ_sr=0.75 is a single sub-sweep added 2026-05-26 to test whether the packed_v1 Magicoder winner transfers.

- **AdamW**: η ∈ {3e-5, 1e-4, 3e-4, 1e-3, 3e-3}
- **chord-tight (polar, k=1)** (`adam-polar-product-lora-coupled-spectral-chord-tight`): η ∈ {3e-3, 1e-2, 3e-2, 1e-1, 3e-1}
- **chord-tight-clean κ_sr=0.75** (`adam-polar-product-lora-coupled-spectral-chord-tight-clean`, polar_method=ssc, ssc_kappa_solver=stable_rank, picard=2): η ∈ {1e-3, 3e-3, 1e-2, 3e-2, 1e-1}

Source log groups: `{adamw,chord_tight}_phase_L_lrsweep_r64_{blackwell,repack_blackwell,extension_blackwell}` + `chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r64_blackwell` (7 groups total). All groups treated as one pool — `packed_v1` and `packed_v1.1` are not distinguished here.

**σ anchor**: no Phase L multi-seed run yet. σ_AdamW(packed_v1, r=64, **4k horizon**) = 0.0017 is the only available number, used here as a *proxy* for unit conversion only. A 9000-step multi-seed AdamW run is needed before σ-units transfer to writeups.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.loader import load_runs
from lora_playground.plotting import compare_variants_figure

# Plain chord-tight k=2 (ns + polar_express) removed from the leaderboard:
# the 2/(ρ·s) cross-coupling coefficient (optim.py:5573) does not benefit
# from picard>1 (packed_v1 r=64 matched-isolation: k=1 0.5120 vs k=3
# 0.5122). Replaced by chord-tight-clean k=2 (1/η coupling, optim.py:5057)
# which on packed_v1 r=64 ns improves k=1 0.5110 → k=3 0.5072.
GROUPS = [
    'adamw_phase_L_lrsweep_r64_blackwell',
    'chord_tight_phase_L_lrsweep_r64_blackwell',
    'adamw_phase_L_lrsweep_r64_repack_blackwell',
    'chord_tight_phase_L_lrsweep_r64_repack_blackwell',
    'adamw_phase_L_lrsweep_r64_extension_blackwell',
    'chord_tight_phase_L_lrsweep_r64_extension_blackwell',
    'chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r64_blackwell',  # κ_sr=0.75 robustness check (k=2)
    'chord_tight_clean_ssc_fixedc_c0p2_phase_L_lrsweep_r64_gpuxl_h200',  # fixed-c=0.2 comparator k=2 (on H200)
    'chord_tight_polar_express_phase_L_lrsweep_r64_gpuxl_h200',  # chord-tight k=1 polar_express @ 10 iters (legacy H200 — walltimed at step 750)
    'chord_tight_polar_express_phase_L_lrsweep_r64_gpuxl_h200_resub',  # chord-tight k=1 polar_express @ 10 iters (H200 resub — completed to step 9000)
    'chord_tight_polar_express_phase_L_lrsweep_r64_lr1e-3_blackwell',  # low-η tail extension (lr=1e-3) for k=1 polar_express — Blackwell (gpuxl 4-GPU-min blocked H200)
    'chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r64_extension_blackwell',  # κ_sr=0.75 LR extension to {3e-1, 1e0}
    'chord_tight_clean_ns_phase_L_lrsweep_r64_k2_blackwell',                # chord-tight-clean k=2 ns (1/η coupling, picard=2)
    'chord_tight_clean_polar_express_phase_L_lrsweep_r64_k2_gpuxl_h200',    # chord-tight-clean k=2 polar_express (1/η coupling, picard=2) — on H200 (matches predecessor 6450610)
    'chord_tight_clean_ssc_kappa075_k1_phase_L_lrsweep_r64_blackwell',      # κ_sr=0.75 k=1 ablation (at k=1 cross-coupling = 0 → polar+SSC)
    'chord_tight_clean_ssc_fixedc_c0p2_k1_phase_L_lrsweep_r64_blackwell',   # c=0.2 k=1 ablation (at k=1 cross-coupling = 0 → polar+SSC)
]

runs = load_runs(where={'log_group': GROUPS},
                 logs_root='../logs', warn_cross_commit=False)

# Dedup: keep the longest-trajectory run per (optimizer, lr, ssc_kappa).
# Phase-L extension runs were resumed from checkpoints, producing two cfg
# events per cell that differ in `resume_from`/`checkpoint_dir`. The loader
# treats them as distinct series; for the leaderboard we want the single
# continuation that reaches the highest step. Adding ssc_kappa to the key
# keeps room for future κ values without collapsing them onto each other.
_dedup = {}
for cfg, evs in runs:
    key = (cfg['optimizer'], float(cfg['lr']), cfg.get('ssc_kappa'), cfg.get('ssc_c'), cfg.get('polar_method'), cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')))
    last_step = evs[-1]['step'] if evs else -1
    prev = _dedup.get(key)
    if prev is None or last_step > prev[1]:
        _dedup[key] = ((cfg, evs), last_step)
runs = [v[0] for v in _dedup.values()]

print(f'loaded: {len(runs)} runs (after dedup)')
for cfg, evs in sorted(runs, key=lambda r: (r[0]['optimizer'], r[0].get('ssc_kappa') or 0, float(r[0]['lr']))):
    last = evs[-1]
    kappa_tag = f"κ={cfg['ssc_kappa']}" if cfg.get('ssc_kappa') is not None else ''
    eff_k = cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override'))
    k_tag = f"k={eff_k}" if eff_k is not None else ''
    print(f"  {cfg['optimizer']:55s} {kappa_tag:9s} {k_tag:4s} η={float(cfg['lr']):.0e}  "
          f"step={last['step']}  eval={last['eval_loss']:.4f}  "
          f"[{cfg.get('log_group', '?')}]")

## Leaderboard + sweep figure

Canonical `compare_variants_figure` panel (matches `packed_v1_leaderboard.ipynb`): left = final eval_loss vs η (log-x); right = best-η trajectory per arm; tables = per-η eval losses + Δ-vs-AdamW summary. `allow_partial=True` shows in-progress runs until they reach step 9000.

In [ ]:
OPT_CT       = 'adam-polar-product-lora-coupled-spectral-chord-tight'
OPT_CT_CLEAN = 'adam-polar-product-lora-coupled-spectral-chord-tight-clean'

def variant_key(cfg):
    opt = cfg.get('optimizer')
    if opt == 'adamw':
        return 'AdamW'
    eff_k = cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override'))
    pm = cfg.get('polar_method')
    # Plain chord-tight (2/(ρ·s) coupling): k=1 only — k=2 dropped because
    # the coefficient does not benefit from picard>1 (see GROUPS comment).
    if opt == OPT_CT and pm == 'polar_express' and eff_k == 1:
        return 'chord-tight k=1 polar_express'
    if opt == OPT_CT and eff_k == 1:
        return 'chord-tight k=1 (ns, polar)'
    # chord-tight-clean SSC arms: split by picard (k=1 ablation vs k=2
    # production). At k=1 the anchored-FW cross-coupling correction is
    # identically zero, so k=1 SSC = polar + SSC clip with no chord-tight
    # cross-coupling (Lemma 4 / §A2 of algorithm_ssc_kappa.md).
    if opt == OPT_CT_CLEAN and cfg.get('ssc_kappa') == 0.75:
        return f'chord-tight-clean κ_sr=0.75 k={eff_k}'
    if opt == OPT_CT_CLEAN and cfg.get('ssc_c') == 0.2:
        return f'chord-tight-clean c=0.2 k={eff_k}'
    # chord-tight-clean k=2 pure 1/η coupling (no ssc_kappa, no ssc_c).
    if (opt == OPT_CT_CLEAN and eff_k == 2
            and cfg.get('ssc_kappa') is None and cfg.get('ssc_c') is None):
        if pm == 'polar_express':
            return 'chord-tight-clean k=2 polar_express'
        return 'chord-tight-clean k=2 (ns)'
    return None

# Pull canonical colors/markers from OPTIM_COLORS / OPTIM_MARKERS so the
# AdamW baseline gets the reserved black overlay (see plotting/colors.py).
# k=1 vs k=2 SSC ablation: distinct hues + distinct markers (compare_variants_figure
# doesn't accept linestyles, so color is the only trajectory-plot signal).
from lora_playground.plotting.colors import OPTIM_COLORS, OPTIM_MARKERS
VARIANT_COLORS = {
    'AdamW':                                  OPTIM_COLORS['adamw'],
    'chord-tight k=1 (ns, polar)':            '#4682b4',  # steelblue
    'chord-tight k=1 polar_express':          '#2ca02c',  # green
    'chord-tight-clean κ_sr=0.75 k=2':        OPTIM_COLORS[OPT_CT_CLEAN],  # dark brown
    'chord-tight-clean κ_sr=0.75 k=1':        '#17becf',  # bright cyan — k=1 ablation, distinct hue
    'chord-tight-clean c=0.2 k=2':            '#ff7f0e',  # vivid orange
    'chord-tight-clean c=0.2 k=1':            '#e377c2',  # hot pink — k=1 ablation, distinct hue
    'chord-tight-clean k=2 (ns)':             '#d62728',  # red — coefficient swap from steelblue
    'chord-tight-clean k=2 polar_express':    '#9467bd',  # purple — coefficient swap from green
}
VARIANT_MARKERS = {
    'AdamW':                                  OPTIM_MARKERS.get('adamw', 's'),
    'chord-tight k=1 (ns, polar)':            OPTIM_MARKERS.get(OPT_CT, 'o'),
    'chord-tight k=1 polar_express':          'P',
    'chord-tight-clean κ_sr=0.75 k=2':        OPTIM_MARKERS.get(OPT_CT_CLEAN, '^'),
    'chord-tight-clean κ_sr=0.75 k=1':        'X',  # heavy X — k=1 ablation
    'chord-tight-clean c=0.2 k=2':            'D',
    'chord-tight-clean c=0.2 k=1':            '*',  # star — k=1 ablation
    'chord-tight-clean k=2 (ns)':             'o',
    'chord-tight-clean k=2 polar_express':    'P',
}

fig, table_df, summary_df = compare_variants_figure(
    variants={
        'AdamW':                                  {},
        'chord-tight k=1 (ns, polar)':            {},
        'chord-tight k=1 polar_express':          {},
        'chord-tight-clean κ_sr=0.75 k=2':        {},
        'chord-tight-clean κ_sr=0.75 k=1':        {},
        'chord-tight-clean c=0.2 k=2':            {},
        'chord-tight-clean c=0.2 k=1':            {},
        'chord-tight-clean k=2 (ns)':             {},
        'chord-tight-clean k=2 polar_express':    {},
    },
    colors=VARIANT_COLORS,
    markers=VARIANT_MARKERS,
    common_where={},  # ignored when prefetched_runs is set
    ref_label='AdamW',
    sigma_ref=0.0017,  # packed_v1 r=64 σ_AdamW @ 4k horizon — PROXY ONLY
    suptitle='OLMo-2-1B × opc-sft-stage2 × r=64 × 9000 steps (Phase L, packed_v1[.1])',
    figsize=(15, 5.5),
    max_steps=9000,
    allow_partial=True,  # κ_sr=0.75 k=1 + c=0.2 k=1 sweeps currently in progress
    final_ylim=(0.72, 0.825),  # crop the diverged AdamW η=3e-3 outlier (1.7074)
    prefetched_runs=runs,
    variant_key=variant_key,
)
from IPython.display import display
display(table_df.style.format('{:.4f}', na_rep='—'))
display(summary_df.style.format({'final': '{:.4f}', 'delta': '{:+.4f}',
                                 'delta_sigma': '{:+.2f}σ', 'best_lr': '{:.0e}'},
                                na_rep='—'))
plt.show()

## r=256 — rank-extension robustness check

Same three arms, same LR grids, same horizon (9000 steps, packed_v1.1). Tests whether the κ_sr=0.75 result transfers from r=64 to r=256 at the same opc-sft-stage2 dataset. Source log groups: `adamw_phase_L_lrsweep_r256_blackwell`, `chord_tight_phase_L_lrsweep_r256_blackwell`, `chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r256_blackwell` (all submitted 2026-05-26; ~6.3 h/task ETA → 24h wall).

σ anchor: same caveat as r=64 — using `0.0017` (packed_v1 r=64 σ_AdamW @ 4k) as a *proxy*. A multi-seed AdamW r=256 9000-step run is needed before σ-units transfer.

In [ ]:
GROUPS_R256 = [
    'adamw_phase_L_lrsweep_r256_blackwell',
    'chord_tight_phase_L_lrsweep_r256_blackwell',
    'chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r256_blackwell',
    'chord_tight_clean_ssc_fixedc_c0p2_phase_L_lrsweep_r256_gpuxl_h200',  # fixed-c=0.2 comparator k=2 (on H200)
    'chord_tight_polar_express_phase_L_lrsweep_r256_gpuxl_h200',  # chord-tight k=1 polar_express @ 10 iters (legacy H200 slow run)
    'chord_tight_polar_express_phase_L_lrsweep_r256_blackwell',  # chord-tight k=1 polar_express batched-Gram (Blackwell)
    'chord_tight_polar_express_phase_L_lrsweep_r256_lr1e-3_blackwell',  # low-η tail extension (lr=1e-3) for k=1 polar_express
    'chord_tight_clean_ssc_kappa075_phase_L_lrsweep_r256_extension_blackwell',  # κ_sr=0.75 LR extension to {3e-1, 1e0}
    'chord_tight_clean_ns_phase_L_lrsweep_r256_k2_blackwell',             # chord-tight-clean k=2 ns (1/η coupling, picard=2)
    'chord_tight_clean_polar_express_phase_L_lrsweep_r256_k2_blackwell',  # chord-tight-clean k=2 polar_express (1/η coupling, picard=2)
    'chord_tight_clean_ssc_kappa075_k1_phase_L_lrsweep_r256_blackwell',   # κ_sr=0.75 k=1 ablation (at k=1 cross-coupling = 0 → polar+SSC)
    'chord_tight_clean_ssc_fixedc_c0p2_k1_phase_L_lrsweep_r256_blackwell', # c=0.2 k=1 ablation (at k=1 cross-coupling = 0 → polar+SSC)
]

runs_r256 = load_runs(where={'log_group': GROUPS_R256},
                      logs_root='../logs', warn_cross_commit=False)

# Same dedup pattern as r=64: keep longest-trajectory per (optimizer, lr, ssc_kappa).
_dedup = {}
for cfg, evs in runs_r256:
    key = (cfg['optimizer'], float(cfg['lr']), cfg.get('ssc_kappa'), cfg.get('ssc_c'), cfg.get('polar_method'), cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')))
    last_step = evs[-1]['step'] if evs else -1
    prev = _dedup.get(key)
    if prev is None or last_step > prev[1]:
        _dedup[key] = ((cfg, evs), last_step)
runs_r256 = [v[0] for v in _dedup.values()]

print(f'loaded: {len(runs_r256)} runs (after dedup)')
for cfg, evs in sorted(runs_r256, key=lambda r: (r[0]['optimizer'], r[0].get('ssc_kappa') or 0, float(r[0]['lr']))):
    last = evs[-1]
    kappa_tag = f"κ={cfg['ssc_kappa']}" if cfg.get('ssc_kappa') is not None else ''
    eff_k = cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override'))
    k_tag = f"k={eff_k}" if eff_k is not None else ''
    print(f"  {cfg['optimizer']:55s} {kappa_tag:9s} {k_tag:4s} η={float(cfg['lr']):.0e}  "
          f"step={last['step']}  eval={last['eval_loss']:.4f}  "
          f"[{cfg.get('log_group', '?')}]")

In [ ]:
fig, table_df_r256, summary_df_r256 = compare_variants_figure(
    variants={
        'AdamW':                                  {},
        'chord-tight k=1 (ns, polar)':            {},
        'chord-tight k=1 polar_express':          {},
        'chord-tight-clean κ_sr=0.75 k=2':        {},
        'chord-tight-clean κ_sr=0.75 k=1':        {},
        'chord-tight-clean c=0.2 k=2':            {},
        'chord-tight-clean c=0.2 k=1':            {},
        'chord-tight-clean k=2 (ns)':             {},
        'chord-tight-clean k=2 polar_express':    {},
    },
    common_where={},  # ignored when prefetched_runs is set
    ref_label='AdamW',
    sigma_ref=0.0017,  # packed_v1 r=64 σ_AdamW @ 4k horizon — PROXY ONLY
    suptitle='OLMo-2-1B × opc-sft-stage2 × r=256 × 9000 steps (Phase L, packed_v1.1)',
    figsize=(15, 5.5),
    max_steps=9000,
    allow_partial=True,  # k=1 SSC ablation sweeps currently in progress
    final_ylim=(0.72, 0.85),  # match r=64 panel; crops diverged AdamW outliers
    prefetched_runs=runs_r256,
    variant_key=variant_key,  # reuses the r=64 cell's variant_key
    colors=VARIANT_COLORS,    # reuses r=64 color overrides
    markers=VARIANT_MARKERS,
)
display(table_df_r256.style.format('{:.4f}', na_rep='—'))
display(summary_df_r256.style.format({'final': '{:.4f}', 'delta': '{:+.4f}',
                                       'delta_sigma': '{:+.2f}σ', 'best_lr': '{:.0e}'},
                                      na_rep='—'))
plt.show()